In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from collections import Counter

## 1. 数据加载与预处理
加载元数据，并将逗号分隔的 `Tags` 拆分、清洗，形成一个整洁的 `(Id, tag)` 长格式表。

In [ ]:
def load_and_clean_tags(file_path, use_cols=['Id', 'Tags']):
    """加载数据并清洗标签。"""
    df = pd.read_csv(file_path, usecols=use_cols, low_memory=False)
    
    # 拆分和分解标签
    tags_df = (df.assign(Tags=df['Tags'].fillna(''))
                 .assign(tag=df['Tags'].str.split(','))
                 .explode('tag'))
    
    # 清洗 tag 文本
    tags_df['tag'] = tags_df['tag'].astype(str).str.strip().str.lower()
    
    # 移除无效标签
    bad_tokens = {'', 'nan', 'none', 'null'}
    tags_df = tags_df[~tags_df['tag'].isin(bad_tokens)]
    
    return tags_df.drop_duplicates(subset=['Id', 'tag']).reset_index(drop=True)

# 执行加载和清洗
file_path = "../data/metadata_merged.csv"
tags_df = load_and_clean_tags(file_path)
print(f"成功加载并处理了 {len(tags_df)} 条 (Id, tag) 记录。")
print(tags_df.head())

## 2. Jaccard 相似度分析与可视化
计算高频标签之间的共现关系（Jaccard相似度），并通过主成分排序优化热力图，使结构更清晰。

In [ ]:
def analyze_jaccard_similarity(tags_df, top_k=40):
    """计算并可视化 Top-K 标签的 Jaccard 相似度。"""
    # 筛选 Top-K 高频标签
    top_tags = tags_df['tag'].value_counts().nlargest(top_k).index.tolist()
    tags_top = tags_df[tags_df['tag'].isin(top_tags)]
    
    # 构造 0/1 指示矩阵 (数据集 × 标签)
    crosstab = pd.crosstab(tags_top['Id'], tags_top['tag'])
    X = (crosstab > 0).astype(np.uint8)
    
    # 计算 Jaccard 相似度矩阵
    co_occurrence = X.T.dot(X).astype(np.int32)
    np.fill_diagonal(co_occurrence.values, 0)
    tag_counts = X.sum(axis=0).to_numpy().astype(np.int32)
    union = tag_counts[:, None] + tag_counts[None, :] - co_occurrence.to_numpy()
    with np.errstate(divide='ignore', invalid='ignore'):
        jaccard_matrix = np.where(union > 0, co_occurrence.to_numpy() / union, 0.0)
    
    # 使用第一主方向对标签排序，使热力图结构更清晰
    eigen_vals, eigen_vecs = np.linalg.eigh((jaccard_matrix + jaccard_matrix.T) / 2.0)
    order = np.argsort(eigen_vecs[:, -1])
    tags_ordered = np.array(top_tags)[order]
    jaccard_ordered = jaccard_matrix[order][:, order]
    
    # 绘制热力图
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(jaccard_ordered, aspect='auto')
    fig.colorbar(im, ax=ax, label='Jaccard 相似度')
    ax.set_xticks(range(len(tags_ordered)))
    ax.set_xticklabels(tags_ordered, rotation=45, ha='right')
    ax.set_yticks(range(len(tags_ordered)))
    ax.set_yticklabels(tags_ordered)
    ax.set_title(f'Top {top_k} 标签共现相似度 (Jaccard, 重排后)')
    plt.tight_layout()
    plt.show()

analyze_jaccard_similarity(tags_df, top_k=40)

## 3. 基于语义的标签聚类 (TF-IDF + KMeans)
使用 TF-IDF 将标签文本向量化，然后通过 KMeans 算法将语义相近的标签分组。

In [ ]:
def semantic_clustering(tags_df, n_clusters=20):
    """对标签进行语义聚类并统计各簇信息。"""
    # 获取唯一的标签列表
    unique_tags = sorted(tags_df['tag'].unique().tolist())
    
    # TF-IDF 向量化
    vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=2)
    X_tfidf = vectorizer.fit_transform(unique_tags)
    
    # KMeans 聚类
    kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
    labels = kmeans.fit_predict(X_tfidf)
    
    # 构建 tag 到 cluster 的映射
    tag_to_cluster = pd.DataFrame({'tag': unique_tags, 'cluster': labels})
    tags_with_clusters = tags_df.merge(tag_to_cluster, on='tag', how='left')
    
    # 统计每个 cluster 覆盖的数据集数量
    cluster_stats = tags_with_clusters.groupby('cluster')['Id'].nunique().sort_values(ascending=False).to_frame('dataset_count')
    
    # 为每个 cluster 找到代表性标签（最常见的几个）
    top_tags_per_cluster = (tags_with_clusters.groupby(['cluster', 'tag'])['Id'].nunique()
                              .sort_values(ascending=False)
                              .groupby(level=0).head(5)
                              .reset_index()
                              .groupby('cluster')['tag'].apply(lambda x: ', '.join(x)))
    
    # 合并结果并打印
    result = cluster_stats.join(top_tags_per_cluster).rename(columns={'tag': 'example_tags'})
    print(f'按语义分为 {n_clusters} 簇，各簇覆盖的数据集数量及代表标签如下：')
    print(result.head(10))
    return X_tfidf, labels

X_tfidf, labels = semantic_clustering(tags_df, n_clusters=25)

## 4. t-SNE 可视化标签簇
使用 t-SNE 对 TF-IDF 向量进行降维，将高维的标签语义空间投影到二维平面上，直观地观察聚类效果。

In [ ]:
def visualize_clusters_tsne(X, labels, perplexity=30):
    """使用 t-SNE 可视化聚类结果。"""
    # t-SNE 降维
    # 为避免性能问题，对大规模数据进行采样
    sample_size = 5000
    if X.shape[0] > sample_size:
        idx = np.random.choice(X.shape[0], sample_size, replace=False)
        X_sample = X[idx]
        labels_sample = labels[idx]
    else:
        X_sample = X
        labels_sample = labels
        
    tsne = TSNE(n_components=2, perplexity=min(perplexity, X_sample.shape[0] - 1), random_state=42, n_iter=300)
    tsne_results = tsne.fit_transform(X_sample.toarray())
    
    # 绘制散点图
    plt.figure(figsize=(12, 10))
    scatter = plt.scatter(tsne_results[:, 0], tsne_results[:, 1], c=labels_sample, cmap='viridis', alpha=0.7)
    plt.title('标签语义聚类 t-SNE 可视化')
    plt.xlabel('t-SNE aixo 1')
    plt.ylabel('t-SNE aixo 2')
    plt.legend(handles=scatter.legend_elements()[0], labels=set(labels_sample))
    plt.grid(True)
    plt.show()

visualize_clusters_tsne(X_tfidf, labels)